# 003 — Subprocess and Bubblewrap

学习目标：

1. 用 asyncio 执行子进程并捕获输出
2. 理解 bubblewrap 的基本原理
3. 手动构建 bwrap 命令，理解每个 namespace flag
4. 用 bwrap 隔离文件系统和进程空间
5. 对比 Clawith `_build_bwrap_command()` 和 `_build_command()`

---


## 1. 基础：用 asyncio 执行代码

先实现最基础的子进程执行——把代码写到临时文件，然后用 Python 运行它。


In [1]:
import asyncio
import tempfile
from pathlib import Path


async def execute_subprocess(code: str, language: str, timeout: int = 10):
    """最简版：在子进程中执行代码，返回 stdout/stderr。"""
    ext = {"python": ".py", "bash": ".sh", "node": ".js"}.get(language)
    if ext is None:
        return f"❌ Unsupported language: {language}"

    # 写临时文件
    tmp = Path(tempfile.mkdtemp()) / f"script{ext}"
    tmp.write_text(code)

    # 构建命令
    cmd_map = {"python": ["python3", str(tmp)], "bash": ["bash", str(tmp)], "node": ["node", str(tmp)]}
    cmd = cmd_map[language]

    # 执行
    proc = await asyncio.create_subprocess_exec(
        *cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )

    try:
        stdout, stderr = await asyncio.wait_for(proc.communicate(), timeout=timeout)
        out = stdout.decode()[:2000]
        err = stderr.decode()[:2000]
        return f"exit={proc.returncode}\nstdout:\n{out}\nstderr:\n{err}" if err else f"exit={proc.returncode}\n{out}"
    except asyncio.TimeoutError:
        proc.kill()
        return "❌ Timeout"
    finally:
        tmp.unlink(missing_ok=True)


# 测试
result = await execute_subprocess("print('Hello from subprocess!')", "python")
print(result)


exit=0
Hello from subprocess!



### 验证隔离——没有隔离会怎样

现在运行一条"危险"命令，看看在没有隔离的情况下会发生什么：


In [2]:
# 安全演示：cat /etc/passwd 在无隔离时可被读取
result = await execute_subprocess("cat /etc/passwd | head -3", "bash")
print(result)


exit=0
root:x:0:0:root:/root:/bin/bash
daemon:x:1:1:daemon:/usr/sbin:/usr/sbin/nologin
bin:x:2:2:bin:/bin:/usr/sbin/nologin



看到了吗？**没有隔离时，子进程可以读取宿主机的任何文件。**  
如果 Agent 被诱导执行 `cat /root/.ssh/id_rsa`，私钥就泄露了。

这就是我们需要 bwrap 的原因。


## 2. bubblewrap 核心概念

bwrap 是 Flatpak 项目开发的工具，用 Linux **user namespaces** 创建轻量级沙盒。

核心思想：
- 创建一个新的 mount namespace
- 用 `--ro-bind` 只读挂载需要的系统目录
- 用 `--bind` 可写挂载工作目录
- 未挂载的路径在沙盒内**不可见**

用图来理解：


In [3]:
print("宿主机文件系统视角：")
print("  /          ← 整个根文件系统")
print("  ├── usr/   ← 系统程序")
print("  ├── etc/   ← 系统配置（含 passwd、shadow）")
print("  ├── home/  ← 用户数据（含 .ssh）")
print("  ├── root/  ← root 用户数据")
print("  └── tmp/   ← 临时文件")
print()
print("bwrap 沙盒内视角（只挂载了 /usr /bin /lib /workspace）：")
print("  /          ← 新的根（只有挂载的内容）")
print("  ├── usr/   ← 只读挂载")
print("  ├── bin/   ← 只读挂载")
print("  ├── lib/   ← 只读挂载")
print("  └── workspace/  ← 可写（工作目录）")
print()
print("  /etc/passwd  → 不存在！")
print("  /home        → 不存在！")
print("  /root        → 不存在！")


宿主机文件系统视角：
  /          ← 整个根文件系统
  ├── usr/   ← 系统程序
  ├── etc/   ← 系统配置（含 passwd、shadow）
  ├── home/  ← 用户数据（含 .ssh）
  ├── root/  ← root 用户数据
  └── tmp/   ← 临时文件

bwrap 沙盒内视角（只挂载了 /usr /bin /lib /workspace）：
  /          ← 新的根（只有挂载的内容）
  ├── usr/   ← 只读挂载
  ├── bin/   ← 只读挂载
  ├── lib/   ← 只读挂载
  └── workspace/  ← 可写（工作目录）

  /etc/passwd  → 不存在！
  /home        → 不存在！
  /root        → 不存在！


## 3. 构建 bwrap 命令

我们来构建一个最小 bwrap 命令，看看隔离效果。


In [4]:
import shutil, subprocess

BWRAP = shutil.which("bwrap")
if not BWRAP:
    print("❌ bwrap 未安装，无法演示")
else:
    print(f"✅ bwrap 路径: {BWRAP}")

# 最小 bwrap 命令：只挂载 /usr 和 /bin，运行 ls /
cmd = [
    BWRAP,
    "--die-with-parent",
    "--new-session",
    "--unshare-user",
    "--unshare-ipc",
    "--unshare-pid",
    "--unshare-uts",
    "--ro-bind", "/usr", "/usr",
    "--ro-bind", "/bin", "/bin",
    "--ro-bind", "/lib", "/lib",
    "--ro-bind", "/lib64", "/lib64",
    "--proc", "/proc",
    "--dev", "/dev",
    "--dir", "/tmp",
    "--",
    "ls", "/"
]

print(f"\n执行的命令:\n{' '.join(cmd)}\n")
r = subprocess.run(cmd, capture_output=True, text=True, timeout=5)
print(f"exit={r.returncode}")
print(f"输出:\n{r.stdout}")
if r.stderr:
    print(f"stderr:\n{r.stderr}")


✅ bwrap 路径: /usr/bin/bwrap

执行的命令:
/usr/bin/bwrap --die-with-parent --new-session --unshare-user --unshare-ipc --unshare-pid --unshare-uts --ro-bind /usr /usr --ro-bind /bin /bin --ro-bind /lib /lib --ro-bind /lib64 /lib64 --proc /proc --dev /dev --dir /tmp -- ls /

exit=0
输出:
bin
dev
lib
lib64
proc
tmp
usr



注意到 `/workspace` 不存在？因为我们没有 `--bind` 工作目录。  
再看看 `/etc/passwd` 是否可读：


In [5]:
if BWRAP:
    cmd = [
        BWRAP,
        "--unshare-user", "--unshare-ipc", "--unshare-pid",
        "--ro-bind", "/usr", "/usr",
        "--ro-bind", "/bin", "/bin",
        "--ro-bind", "/lib", "/lib",
        "--ro-bind", "/lib64", "/lib64",
        "--proc", "/proc", "--dev", "/dev", "--dir", "/tmp",
        "--",
        "cat", "/etc/passwd"
    ]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=5)
    if r.returncode != 0:
        print(f"✅ 隔离成功！cat /etc/passwd 失败: {r.stderr.strip()}")
    else:
        print(f"❌ 未隔离: {r.stdout[:100]}")


✅ 隔离成功！cat /etc/passwd 失败: cat: /etc/passwd: No such file or directory


## 4. 加入工作目录

我们挂载一个可写的工作目录，让沙盒能执行代码并写入结果。


In [6]:
import tempfile, os

if BWRAP:
    work_dir = Path(tempfile.mkdtemp())
    script = work_dir / "hello.py"
    script.write_text("import os\nprint('工作目录:', os.getcwd())\nprint('目录内容:', os.listdir('/workspace'))")

    cmd = [
        BWRAP,
        "--unshare-user", "--unshare-ipc", "--unshare-pid",
        "--ro-bind", "/usr", "/usr",
        "--ro-bind", "/bin", "/bin",
        "--ro-bind", "/lib", "/lib",
        "--ro-bind", "/lib64", "/lib64",
        "--proc", "/proc", "--dev", "/dev", "--dir", "/tmp",
        "--bind", str(work_dir), "/workspace",   # 工作目录可写挂载
        "--setenv", "HOME", "/workspace",
        "--chdir", "/workspace",
        "--",
        "python3", "/workspace/hello.py"
    ]

    print(f"工作目录: {work_dir}")
    print(f"执行的命令:\n{' '.join(str(c) for c in cmd)}\n")
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=5)
    print(f"exit={r.returncode}")
    print(f"stdout:\n{r.stdout}")
    if r.stderr:
        print(f"stderr:\n{r.stderr}")


工作目录: /tmp/tmpnd6nw60u
执行的命令:
/usr/bin/bwrap --unshare-user --unshare-ipc --unshare-pid --ro-bind /usr /usr --ro-bind /bin /bin --ro-bind /lib /lib --ro-bind /lib64 /lib64 --proc /proc --dev /dev --dir /tmp --bind /tmp/tmpnd6nw60u /workspace --setenv HOME /workspace --chdir /workspace -- python3 /workspace/hello.py

exit=0
stdout:
工作目录: /workspace
目录内容: ['hello.py']



## 5. 网络隔离

`--unshare-net` 可以隔离网络 namespace。沙盒内的进程无法访问网络。


In [7]:
if BWRAP:
    def test_network(network_flag: list[str]) -> str:
        """在沙盒内测试网络是否可达。"""
        # 用 hostname -I 或 curl 测试网络
        cmd = [
            BWRAP,
            "--unshare-user", "--unshare-ipc", "--unshare-pid",
            "--ro-bind", "/usr", "/usr",
            "--ro-bind", "/bin", "/bin",
            "--ro-bind", "/lib", "/lib",
            "--ro-bind", "/lib64", "/lib64",
            "--proc", "/proc", "--dev", "/dev", "--dir", "/tmp",
            "--dir", "/workspace",
        ] + network_flag + [
            "--",
            "python3", "-c", 
            "import urllib.request; "
            "r = urllib.request.urlopen('http://example.com', timeout=3); "
            "print('Network OK:', r.status)"
        ]
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
            return f"exit={r.returncode}, stdout={r.stdout[:100].strip()}"
        except subprocess.TimeoutExpired:
            return "❌ Timeout"
        except Exception as e:
            return f"❌ {e}"

    print("🛑 有网络隔离 (--unshare-net):")
    print("   ", test_network(["--unshare-net"]))

    print()
    print("🌐 无网络隔离:")
    # 没有 --unshare-net，而且需要 bind mount /etc 才能解析 DNS
    # 这里简化，只演示概念
    print("   跳过（需要 mount /etc/resolv.conf，后面会展示）")


🛑 有网络隔离 (--unshare-net):
    exit=1, stdout=

🌐 无网络隔离:
   跳过（需要 mount /etc/resolv.conf，后面会展示）


## 6. 对比 Clawith 的实现

Clawith 的 `SubprocessBackend._build_bwrap_command()` 构建了完整的 bwrap 参数。
打开源码看关键部分：


In [8]:
print("=" * 60)
print("Clawith _build_bwrap_command 关键参数对照")
print("=" * 60)
print()

flags = [
    ("--die-with-parent", "父进程退出时子进程自动终止"),
    ("--new-session", "创建新的 session"),
    ("--unshare-user", "隔离 UID/GID 映射"),
    ("--unshare-ipc", "隔离 System V IPC"),
    ("--unshare-pid", "隔离进程 ID 空间"),
    ("--unshare-uts", "隔离 hostname"),
    ("--unshare-cgroup", "隔离 cgroup 层次"),
    ("--ro-bind /usr /usr", "只读挂载系统程序"),
    ("--ro-bind /bin /bin", "只读挂载系统程序"),
    ("--ro-bind /lib /lib", "只读挂载系统库"),
    ("--ro-bind /lib64 /lib64", "只读挂载 64 位系统库"),
    ("--ro-bind /etc /etc", "只读挂载系统配置（含 DNS）"),
    ("--bind <work> /workspace", "可写挂载工作目录"),
    ("--dev /dev", "提供设备节点"),
    ("--proc /proc", "提供进程信息"),
    ("--dir /tmp", "提供空 /tmp 目录"),
    ("--setenv HOME /workspace", "隔离 HOME 环境变量"),
    ("--chdir /workspace", "设置工作目录"),
    ("--unshare-net (条件)", "allow_network=False 时隔离网络"),
]

for flag, desc in flags:
    print(f"  {flag:35s}  ← {desc}")
print()
print("源码位置: subprocess_backend.py:178-230")


Clawith _build_bwrap_command 关键参数对照

  --die-with-parent                    ← 父进程退出时子进程自动终止
  --new-session                        ← 创建新的 session
  --unshare-user                       ← 隔离 UID/GID 映射
  --unshare-ipc                        ← 隔离 System V IPC
  --unshare-pid                        ← 隔离进程 ID 空间
  --unshare-uts                        ← 隔离 hostname
  --unshare-cgroup                     ← 隔离 cgroup 层次
  --ro-bind /usr /usr                  ← 只读挂载系统程序
  --ro-bind /bin /bin                  ← 只读挂载系统程序
  --ro-bind /lib /lib                  ← 只读挂载系统库
  --ro-bind /lib64 /lib64              ← 只读挂载 64 位系统库
  --ro-bind /etc /etc                  ← 只读挂载系统配置（含 DNS）
  --bind <work> /workspace             ← 可写挂载工作目录
  --dev /dev                           ← 提供设备节点
  --proc /proc                         ← 提供进程信息
  --dir /tmp                           ← 提供空 /tmp 目录
  --setenv HOME /workspace             ← 隔离 HOME 环境变量
  --chdir /workspace                   ← 设置工作目录
  --unshare-net (条件)  

---

**小结：** bwrap 通过 Linux namespace 实现了轻量级的文件系统、网络和进程隔离。  
它是 Clawith 默认沙盒的核心隔离层。

下一份 Notebook 讲**另一层防御**：静态代码扫描和资源限制。
